<a href="https://colab.research.google.com/github/ep24b009-HariccharanM/Coding-Exercises/blob/main/Variational_Quantum_Eigensolver_4x4/VQE_4x4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit[visualization] qiskit-ibm-runtime qiskit-aer qiskit_qasm3_import qiskit-algorithms

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter, ParameterVector
import qiskit.qasm3
from qiskit_ibm_runtime.fake_provider import FakeVigoV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import *
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.primitives import StatevectorSampler, PrimitiveJob
import matplotlib.pyplot as plt
import math

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.2/108.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.0/218.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━

In [52]:
import numpy as np

# Given matrix
Herm = np.array([[2,0,0,0],[0,5,0,0],[0,0,0,1],[0,0,1,0]])

# Decomposing

I = np.eye(2)
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])

a = np.zeros(16 , dtype = complex)

for i in range(0,4):
  for j in range(0,4):
    if i == 0:
        A = I
    elif i == 1:
        A = X
    elif i == 2:
        A = Y
    else:
        A = Z

    if j == 0:
        B = I
    elif j == 1:
        B = X
    elif j == 2:
        B = Y
    else:
        B = Z

    a[i*4+j]= 0.25* np.trace(Herm @ ( np.kron(B,A)))

print(a)

#Initializing the wavefunction with 8 parameters
p = np.random.rand(8)*2*np.pi

[ 1.75+0.j  0.  +0.j  0.  +0.j  1.75+0.j  0.5 +0.j  0.  +0.j  0.  +0.j
 -0.5 +0.j  0.  +0.j  0.  +0.j  0.  +0.j  0.  +0.j -0.75+0.j  0.  +0.j
  0.  +0.j -0.75+0.j]


In [53]:
def anz(p):

  v = np.zeros((9,4))

  for i in range(9):
    qc = QuantumCircuit(2,2)
    qc.ry(float(np.real(p[0])),0)
    qc.rz(float(np.real(p[1])),0)

    qc.ry(float(np.real(p[2])),1)
    qc.rz(float(np.real(p[3])),1)

    qc.cx(0,1)

    qc.ry(float(np.real(p[4])),0)
    qc.rz(float(np.real(p[5])),0)

    qc.ry(float(np.real(p[6])),1)
    qc.rz(float(np.real(p[7])),1)

    if i==0:
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==1:
      qc.h(1)
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==2:
      qc.sdg(1)
      qc.h(1)
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==3:
      qc.h(0)
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==4:
      qc.h(0)
      qc.h(1)
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==5:
      qc.h(0)
      qc.sdg(1)
      qc.h(1)
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==6:
      qc.sdg(0)
      qc.h(0)
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==7:
      qc.sdg(0)
      qc.h(0)
      qc.h(1)
      qc.measure(0,0)
      qc.measure(1,1)
    elif i==8:
      qc.sdg(0)
      qc.h(0)
      qc.sdg(1)
      qc.h(1)
      qc.measure(0,0)
      qc.measure(1,1)

    simulator = AerSimulator()
    result = simulator.run(qc, shots=1024).result()
    counts = result.get_counts()

    b1 = counts.get("00", 0)
    b2 = counts.get("10", 0)
    b3 = counts.get("01", 0)
    b4 = counts.get("11", 0)

    v[i,0] = b1/1024
    v[i,1] = b2/1024
    v[i,2] = b3/1024
    v[i,3] = b4/1024

  return v

In [54]:
ord = {1:1, 2:2, 3:0, 4:3, 5:4, 6:5, 7:3, 8:6, 9:7, 10:8, 11:6, 12:0, 13:1, 14:2, 15:0 }

def mult(v,i,m):
  y=0
  if m==0:
    y = v[i,0] - v[i,1] + v[i,2] - v[i,3]
  elif m==1:
    y = v[i,0] + v[i,1] - v[i,2] - v[i,3]
  elif m==2:
    y = v[i,0] - v[i,1] - v[i,2] + v[i,3]

  return y

def cost(p,a): # The cost function that is to be minimized.

  v = anz(p)
  c = a[0]

  for i in range(1,16):
    if i == 1 or i == 2 or i == 3 :
      c += a[i]*mult(v,ord[i],0)
    elif i == 4 or i == 8 or i == 12 :
      c += a[i]*mult(v,ord[i],1)
    else :
      c += a[i]*mult(v,ord[i],2)

  return np.real(c)

# **Gradient Method**

In [55]:
lr = 0.1
steps = 200

def grad(p,a):
  gr = np.zeros(8, dtype=complex)
  for i in range(8):
    p_plus = p.copy()
    p_minus = p.copy()

    p_plus[i] = (np.real(p_plus[i]) + lr) % (2 * np.pi)
    p_minus[i] = (np.real(p_minus[i]) - lr) % (2 * np.pi)

    gr[i] = (cost(p_plus, a) - cost(p_minus, a))/(2*lr)
  return gr

for j in range(steps):
  gr = grad(p,a)
  p = p - lr*gr
  p = np.real(p) % (2 * np.pi)

print("The optimum parameters are",p)

The optimum parameters are [3.14436479e+00 6.21286427e+00 1.66988354e-02 1.50110467e+00
 1.56420663e+00 4.51128153e-03 7.69607012e-03 6.08435100e+00]


In [56]:
qc = QuantumCircuit(2,2)
qc.ry(float(np.real(p[0])),0)
qc.rz(float(np.real(p[1])),0)

qc.ry(float(np.real(p[2])),1)
qc.rz(float(np.real(p[3])),1)

qc.cx(0,1)

qc.ry(float(np.real(p[4])),0)
qc.rz(float(np.real(p[5])),0)

qc.ry(float(np.real(p[6])),1)
qc.rz(float(np.real(p[7])),1)

state = Statevector.from_instruction(qc)

print("Exact Statevector:")
print(state.data) # For reference the required state vector is [0,0,0.707,-0.707]

Exact Statevector:
[-0.00275357-0.00542555j  0.00113943+0.0066065j  -0.44503117+0.54645863j
  0.45043258-0.54804996j]


# **COBYLA Method to optimise**

In [57]:
from scipy.optimize import minimize

def run_classical_optimization(initial_p, coefficients):
    """
    Takes an initial parameter array and coefficient array,
    and optimizes p to minimize the cost_function.
    """
    print("Starting optimization...")

    result = minimize(
        fun=cost,
        x0=initial_p,
        args=(coefficients,),
        method='COBYLA',
        options={'maxiter': 1000, 'disp': True}
    )


    optimized_p = result.x
    minimum_cost = result.fun

    return optimized_p, minimum_cost

p_f , c_min = run_classical_optimization(p, a)

Starting optimization...
Return from COBYLA because the trust region radius reaches its lower bound.
Number of function values = 40   Least value of F = -1.0
The corresponding X is:
[3.14436479e+00 7.21286427e+00 1.66988354e-02 1.50110467e+00
 1.56420663e+00 4.51128153e-03 7.69607012e-03 6.08435100e+00]



In [58]:
qc = QuantumCircuit(2,2)
qc.ry(float(np.real(p_f[0])),0)
qc.rz(float(np.real(p_f[1])),0)

qc.ry(float(np.real(p_f[2])),1)
qc.rz(float(np.real(p_f[3])),1)

qc.cx(0,1)

qc.ry(float(np.real(p_f[4])),0)
qc.rz(float(np.real(p_f[5])),0)

qc.ry(float(np.real(p_f[6])),1)
qc.rz(float(np.real(p_f[7])),1)

state = Statevector.from_instruction(qc)

print("Exact Statevector:") # For reference the required state vector is [0,0,0.707,-0.707]
print(state.data)

Exact Statevector:
[ 0.00073114-0.00531321j -0.00162795+0.00710971j -0.65254015+0.26621184j
  0.65803857-0.26500169j]


Hence both the Gradient method and COBYLA method have converged to the required State vector.